# 1. Imports

In [1]:
import torchaudio
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import panphon
import panphon.distance

import csv
import os
import requests
import json
import re
from jiwer import wer

from datasets import Dataset, Audio
from nltk.metrics.distance import edit_distance, edit_distance_align
import numpy as np
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from pathlib import Path

from tqdm import tqdm

/home/manon/Documents/Master/M2/stage/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports from scripts

In [2]:
from models.w2v2p import load_model, predict, predict_from_dataset, predict_simple
from data.get_data import get_sentences, get_phonemes_from_LC, get_phonemes_from_json, get_x, get_ref
from data.form_dataset import make_dataset
from data.make_groups import regroupement, regroupement_mixte, split_variantes
from data.get_closest import closest
from data.cleaning import clean_text, clean_list
from evaluation.metrics import get_per_nltk, per_by_variant, per_aligne_nltk, per_aligne_by_var, get_pfer_panphon, pfer_var
from evaluation.pipeline import evaluate_dataset

# 2. Loading paths

In [3]:
base_path = Path("/home/manon/Documents/Master/M2/stage")
data_path = base_path / "data"
audio_path = base_path / "data/common_voice_oc/clips/"

# 3. Loading the model

In [4]:
processor, model = load_model("facebook/wav2vec2-xlsr-53-espeak-cv-ft")

Loading weights: 100%|██████████| 424/424 [00:01<00:00, 367.06it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


# 4. Loading data

In [5]:
y_train = get_phonemes_from_json(base_path / f"phonemes_train.json")
y_test = get_phonemes_from_json(base_path / f"phonemes_test.json")

X_train = get_x("train")
X_test = get_x("test")

# 5. Build dataset

In [6]:
test_dataset = make_dataset(X_test, y_test)

# 6. Run inference

Test sur un audio

In [7]:
audio = (audio_path / f"common_voice_oc_36672373.mp3")

prediction = predict_simple(model, processor, audio)
print(prediction)

['k e s o ɾ a v ɔ s t ɐ ð i s p o z i s j o n d i ɾ ɛ k']


Test sur plusieurs audios

In [8]:
audios = [test_dataset["audio"][i]["array"] for i in range(10)]

predictions = []
for i, audio in enumerate(tqdm(audios)):
    path = X_test[i]
    pred = predict_simple(model, processor, path)
    predictions.extend(pred)

print(predictions)

100%|██████████| 10/10 [00:15<00:00,  1.57s/it]

['k e s o ɾ a v ɔ s t ɐ ð i s p o z i s j o n d i ɾ ɛ k', 'e n a r e s m u m e n k ɾ e v ɛ t u m i o a r a ɲ a k aʊ ʃ i ɡ a d a', 'a v aɪ ɾ a k e l i s e ɾ a i z i t dʒ e s e ð a s i ɾ u', 'e n t ɾ e ɾ ə n e n l a s a l a ð e s p a ɾ e t ts i n k l i n a ð e s e ð e t ts e l u β ɛ r', 'e d a s a m a r o k a k e n u s e m b u r l a ɾ i e m e s', 'd e v e r t a n u j a r e z ə n t a m e t e', 'e r a s ɛ b n ə', 'k ɛ l i d i d ɛ k', 'p e r d i w a m a n', 'd e p i k s e β i ɾ ɛ k e k w o j e k e ɾ a p i p a d a m f o r s a']


# 7. Clean predictions

In [9]:
true_clean, pred_clean = clean_list(y_test, list_of_lists=True), clean_list(predictions)

Keep only closest variant from LC for each sentence

In [10]:
true_closest = closest(pred_clean, true_clean)

# 7. Evaluate

### Compute PER (Phoneme Error Rate)

In [11]:
per = get_per_nltk(true_closest, pred_clean, "test")
print("PER : ", per)

PER :  32.17%


# 8. Test de la pipeline sur tout le dataset

In [12]:
per = evaluate_dataset(
    test_dataset,
    y_test,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    X_test,
)

print(f"PER sur tout le dataset: {per}")

100%|██████████| 274/274 [07:28<00:00,  1.64s/it]

PER sur tout le dataset: (['k', 'e', 's', 'ɔ', 'a', 'ʀ', 'ɔ', 'v', 'ɔ', 's', 't', 'ɔ', 'd', 'i', 's', 'p', 'u', 'z', 'i', 's', 'j', 'u', 'n', 'd', 'i', 'd', 'e'], ['d', 'ʒ', 'u', 'b', 'a', 's', 't', 'i', 's', 'k', 'i', 'y', 'a', 'k', 'a', 'z', 'a', 'e', 'p', 'l', 'a', 'n', 't', 'i', 'd', 'ʒ', 'a', 'r', 'ð', 'i', 's'], [0.8709677419354839], '87.10%')
